# Bulk File Research Tool

This notebook ingests a large number of files (PDFs, URLs, text) in bulk, applies systematic AI-powered extraction via reusable transformation templates, generates embeddings for semantic search, synthesizes cross-document findings, and optionally produces a podcast audio discussion of the results.

**Pipeline:** File Discovery → Text Extraction → Transformation Templates → Batch AI Extraction → Embedding + Search → Cross-Document Analysis → Export → Podcast

**Patterns adopted from [open-notebook](open-notebook/):** transformation system, batch embedding with retry, text chunking, multi-provider LLM, parallel execution with semaphore, podcast generation.

In [ ]:
import os
import json
import time
import asyncio
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

from config import LLMConfig, LLMClient

# LLM setup
llm_config = LLMConfig.from_env()
llm_client = LLMClient(llm_config)

# Embedding config
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
EMBEDDING_BATCH_SIZE = int(os.getenv("EMBEDDING_BATCH_SIZE", "50"))
CHUNK_SIZE = 400       # tokens per chunk
CHUNK_OVERLAP = 60     # token overlap between chunks
MAX_CONCURRENT_LLM = 5 # parallel LLM call limit

# TTS config
TTS_PROVIDER = os.getenv("TTS_PROVIDER", "openai")
TTS_VOICE = os.getenv("TTS_VOICE", "alloy")

# Paths
INPUT_DIR = Path(os.getenv("RESEARCH_INPUT_DIR", "./research_input"))
OUTPUT_DIR = Path(os.getenv("RESEARCH_OUTPUT_DIR", "./research_output"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"LLM Provider: {llm_config.provider} | Model: {llm_config.model}")
print(f"Embedding:    {EMBEDDING_MODEL} | Batch: {EMBEDDING_BATCH_SIZE}")
print(f"Chunking:     {CHUNK_SIZE} tokens, {CHUNK_OVERLAP} overlap")
print(f"TTS:          {TTS_PROVIDER} | Voice: {TTS_VOICE}")
print(f"Input:        {INPUT_DIR.resolve()}")
print(f"Output:       {OUTPUT_DIR.resolve()}")
print(f"Concurrency:  {MAX_CONCURRENT_LLM} parallel LLM calls")

In [ ]:
import csv

SUPPORTED_EXTENSIONS = {".pdf", ".txt", ".md", ".html", ".htm"}

manifest = []

# Scan local files
for f in sorted(INPUT_DIR.rglob("*")):
    if f.is_file() and f.suffix.lower() in SUPPORTED_EXTENSIONS:
        manifest.append({
            "path": str(f),
            "type": f.suffix.lower().lstrip("."),
            "title": f.stem,
            "size_bytes": f.stat().st_size,
        })

# Check for urls.csv (columns: url, title)
urls_csv = INPUT_DIR / "urls.csv"
if urls_csv.exists():
    with open(urls_csv, newline="", encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            manifest.append({
                "path": row["url"].strip(),
                "type": "url",
                "title": row.get("title", row["url"]).strip(),
                "size_bytes": 0,
            })

# Print summary
type_counts = {}
for entry in manifest:
    type_counts[entry["type"]] = type_counts.get(entry["type"], 0) + 1

print(f"Discovered {len(manifest)} sources:")
for t, c in sorted(type_counts.items()):
    print(f"  {t:6s} : {c}")
print()
for i, entry in enumerate(manifest[:20]):
    size = f"{entry['size_bytes']:>10,} B" if entry['size_bytes'] else "       URL"
    print(f"  [{i:3d}] {entry['type']:4s} {size}  {entry['title'][:60]}")
if len(manifest) > 20:
    print(f"  ... and {len(manifest) - 20} more")

In [ ]:
import tiktoken
from tqdm.notebook import tqdm

enc = tiktoken.encoding_for_model("gpt-4o")

def extract_text_from_pdf(path: str) -> str:
    """Extract text via Marker with PyPDF2 fallback."""
    try:
        from marker.converters.pdf import PdfConverter
        from marker.models import create_model_dict
        from marker.config.parser import ConfigParser
        from marker.output import text_from_rendered
        config_parser = ConfigParser({"output_format": "markdown"})
        converter = PdfConverter(config=config_parser.generate_config_dict(), artifact_dict=create_model_dict())
        rendered = converter(path)
        text, _, _ = text_from_rendered(rendered)
        if text.strip():
            return text
    except Exception:
        pass
    # Fallback
    from PyPDF2 import PdfReader
    reader = PdfReader(path)
    return "\n".join(page.extract_text() or "" for page in reader.pages)


def extract_text_from_url(url: str) -> str:
    import requests
    from bs4 import BeautifulSoup
    resp = requests.get(url, timeout=30, headers={"User-Agent": "ResearchBot/1.0"})
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()
    return soup.get_text(separator="\n", strip=True)


def extract_text_from_file(path: str) -> str:
    return Path(path).read_text(encoding="utf-8", errors="replace")


EXTRACTORS = {
    "pdf": extract_text_from_pdf,
    "txt": extract_text_from_file,
    "md": extract_text_from_file,
    "html": extract_text_from_file,
    "htm": extract_text_from_file,
    "url": extract_text_from_url,
}

documents = []
failed = []

for entry in tqdm(manifest, desc="Extracting text"):
    try:
        text = EXTRACTORS[entry["type"]](entry["path"])
        tokens = len(enc.encode(text))
        documents.append({
            "title": entry["title"],
            "source": entry["path"],
            "type": entry["type"],
            "text": text,
            "char_count": len(text),
            "token_estimate": tokens,
        })
    except Exception as e:
        failed.append({"title": entry["title"], "error": str(e)})

total_tokens = sum(d["token_estimate"] for d in documents)
print(f"\nExtracted: {len(documents)} succeeded, {len(failed)} failed")
print(f"Total tokens: {total_tokens:,}")
if failed:
    print("\nFailed sources:")
    for f in failed:
        print(f"  - {f['title']}: {f['error'][:80]}")

In [ ]:
from pydantic import BaseModel
from typing import Optional, List


class TransformationTemplate(BaseModel):
    name: str
    title: str
    description: str
    prompt: str
    model_id: Optional[str] = None


templates: List[TransformationTemplate] = [
    TransformationTemplate(
        name="summary",
        title="Summary",
        description="Concise 3-paragraph summary of the document",
        prompt=(
            "Write a concise 3-paragraph summary of the following document. "
            "Paragraph 1: main topic and context. "
            "Paragraph 2: key methods or arguments. "
            "Paragraph 3: results and significance.\n\n{text}"
        ),
    ),
    TransformationTemplate(
        name="methodology",
        title="Methodology",
        description="Extract research methodology and approach",
        prompt=(
            "Extract and describe the research methodology from this document. "
            "Include: approach type, data sources, experimental setup, evaluation metrics, "
            "and any baselines compared against. If not a research paper, describe the "
            "analytical framework used.\n\n{text}"
        ),
    ),
    TransformationTemplate(
        name="key_findings",
        title="Key Findings",
        description="List key findings and contributions",
        prompt=(
            "List the key findings and contributions of this document as numbered points. "
            "For each finding, include: the claim, supporting evidence, and significance. "
            "Limit to the 5-10 most important findings.\n\n{text}"
        ),
    ),
    TransformationTemplate(
        name="gaps_limitations",
        title="Gaps & Limitations",
        description="Identify gaps, limitations, and future work",
        prompt=(
            "Identify the gaps, limitations, and suggested future work from this document. "
            "Include: acknowledged limitations, unstated assumptions, methodological weaknesses, "
            "and open questions. Be critical but fair.\n\n{text}"
        ),
    ),
    TransformationTemplate(
        name="citations_refs",
        title="Key References",
        description="Extract key references and how they are used",
        prompt=(
            "Extract the most important references cited in this document. For each, state: "
            "the referenced work, how it's used (foundation, comparison, critique), and why "
            "it matters to this document's argument. Limit to top 10 references.\n\n{text}"
        ),
    ),
]

# -- Add custom templates here --
# templates.append(TransformationTemplate(
#     name="custom", title="Custom", description="...",
#     prompt="Your prompt here...\n\n{text}"
# ))

print(f"Defined {len(templates)} transformation templates:")
for t in templates:
    print(f"  - {t.name}: {t.description}")

In [ ]:
from tenacity import retry, stop_after_attempt, wait_exponential

semaphore = asyncio.Semaphore(MAX_CONCURRENT_LLM)


@retry(stop=stop_after_attempt(3), wait=wait_exponential(min=2, max=30))
async def run_transform(doc: dict, template: TransformationTemplate) -> dict:
    prompt_text = template.prompt.format(text=doc["text"][:50000])  # cap input
    async with semaphore:
        start = time.time()
        # Run sync LLM call in thread pool to not block event loop
        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(
            None,
            llm_client.chat,
            [{"role": "user", "content": prompt_text}],
        )
        duration = time.time() - start
    return {
        "doc_title": doc["title"],
        "template_name": template.name,
        "output": result,
        "model": llm_config.model,
        "tokens_used": len(enc.encode(prompt_text)) + len(enc.encode(result)),
        "duration_s": round(duration, 1),
    }


async def batch_transform_all():
    tasks = []
    for doc in documents:
        for tmpl in templates:
            tasks.append(run_transform(doc, tmpl))

    results = []
    errors = []
    pbar = tqdm(total=len(tasks), desc="Transforming")
    for coro in asyncio.as_completed(tasks):
        try:
            r = await coro
            results.append(r)
        except Exception as e:
            errors.append(str(e))
        pbar.update(1)
    pbar.close()
    return results, errors


start_time = time.time()
transformations, transform_errors = await batch_transform_all()
elapsed = time.time() - start_time

total_t_tokens = sum(t["tokens_used"] for t in transformations)
print(f"\nCompleted: {len(transformations)} transformations in {elapsed:.0f}s")
print(f"Failed:    {len(transform_errors)}")
print(f"Tokens:    {total_t_tokens:,}")
print(f"Avg time:  {sum(t['duration_s'] for t in transformations) / max(len(transformations), 1):.1f}s per call")

In [ ]:
import numpy as np
from openai import OpenAI

embed_client = OpenAI(
    api_key=llm_config.api_key,
    base_url=llm_config.base_url if "embedding" not in llm_config.base_url else llm_config.base_url,
)


def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Split text into token-sized chunks on sentence boundaries."""
    tokens = enc.encode(text)
    if len(tokens) <= chunk_size:
        return [text]
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_str = enc.decode(chunk_tokens)
        # Try to break on sentence boundary
        last_period = chunk_str.rfind(". ")
        if last_period > len(chunk_str) // 2 and end < len(tokens):
            chunk_str = chunk_str[: last_period + 1]
            end = start + len(enc.encode(chunk_str))
        chunks.append(chunk_str.strip())
        start = end - overlap
    return [c for c in chunks if len(enc.encode(c)) >= 5]


@retry(stop=stop_after_attempt(3), wait=wait_exponential(min=2, max=30))
def embed_batch(texts: list[str]) -> list[list[float]]:
    resp = embed_client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in resp.data]


# Chunk all documents
all_chunks = []  # {doc_idx, chunk_idx, text, doc_title}
for doc_idx, doc in enumerate(documents):
    for chunk_idx, chunk in enumerate(chunk_text(doc["text"])):
        all_chunks.append({
            "doc_idx": doc_idx,
            "chunk_idx": chunk_idx,
            "text": chunk,
            "doc_title": doc["title"],
        })

print(f"Total chunks: {len(all_chunks)}")

# Embed in batches
all_embeddings = []
for i in tqdm(range(0, len(all_chunks), EMBEDDING_BATCH_SIZE), desc="Embedding"):
    batch_texts = [c["text"] for c in all_chunks[i : i + EMBEDDING_BATCH_SIZE]]
    batch_embs = embed_batch(batch_texts)
    all_embeddings.extend(batch_embs)

# Build FAISS index (cosine similarity via L2-normalized IndexFlatIP)
import faiss

emb_matrix = np.array(all_embeddings, dtype="float32")
faiss.normalize_L2(emb_matrix)
dimension = emb_matrix.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(emb_matrix)

faiss.write_index(index, str(OUTPUT_DIR / "search_index.faiss"))

print(f"Index built: {index.ntotal} vectors, {dimension} dimensions")
print(f"Saved to {OUTPUT_DIR / 'search_index.faiss'}")

In [ ]:
def search(query: str, top_k: int = 10, min_score: float = 0.2) -> list[dict]:
    """Semantic search over all document chunks."""
    q_emb = np.array(embed_batch([query])[0], dtype="float32").reshape(1, -1)
    faiss.normalize_L2(q_emb)
    scores, indices = index.search(q_emb, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx < 0 or score < min_score:
            continue
        chunk = all_chunks[idx]
        results.append({
            "doc_title": chunk["doc_title"],
            "chunk_text": chunk["text"][:500],
            "score": round(float(score), 4),
            "doc_idx": chunk["doc_idx"],
            "chunk_idx": chunk["chunk_idx"],
        })
    return results


def ask(question: str, top_k: int = 5) -> dict:
    """RAG: search + LLM synthesis with citations."""
    hits = search(question, top_k=top_k)
    if not hits:
        return {"answer": "No relevant sources found.", "sources": []}
    context = "\n\n".join(
        f"[{h['doc_title']}] (score {h['score']}):\n{h['chunk_text']}"
        for h in hits
    )
    prompt = (
        f"Based on the following research excerpts, answer the question. "
        f"Cite sources by [title] when making claims.\n\n"
        f"EXCERPTS:\n{context}\n\nQUESTION: {question}"
    )
    answer = llm_client.chat([{"role": "user", "content": prompt}])
    return {
        "answer": answer,
        "sources": [{"doc_title": h["doc_title"], "score": h["score"],
                      "snippet": h["chunk_text"][:200]} for h in hits],
    }


# Demo search
if documents:
    demo_results = search("main methodology and approach", top_k=5)
    print("Demo search: 'main methodology and approach'\n")
    for r in demo_results:
        print(f"  [{r['score']:.3f}] {r['doc_title']}: {r['chunk_text'][:100]}...")
else:
    print("No documents loaded — add files to research_input/ and re-run.")

In [ ]:
# Group transformation outputs by template
by_template: dict[str, list[dict]] = {}
for t in transformations:
    by_template.setdefault(t["template_name"], []).append(t)


async def synthesize_template(name: str, outputs: list[dict]) -> str:
    combined = "\n\n---\n\n".join(
        f"## {o['doc_title']}\n{o['output']}" for o in outputs
    )
    prompt = (
        f"You are analyzing {len(outputs)} documents' '{name}' extractions.\n"
        f"Synthesize across all documents:\n"
        f"1. Common themes and patterns\n"
        f"2. Contradictions or disagreements\n"
        f"3. Knowledge gaps across documents\n"
        f"4. Suggested clustering/categorization\n\n"
        f"DOCUMENT EXTRACTIONS:\n{combined[:40000]}"
    )
    async with semaphore:
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(
            None, llm_client.chat, [{"role": "user", "content": prompt}]
        )


analysis = {}
for name, outputs in tqdm(by_template.items(), desc="Cross-document analysis"):
    analysis[name] = await synthesize_template(name, outputs)

print("\n" + "=" * 60)
for name, text in analysis.items():
    print(f"\n### {name.upper()} SYNTHESIS\n")
    print(text[:500])
    if len(text) > 500:
        print(f"\n  ... ({len(text)} chars total)")
    print()

In [ ]:
# Save transformations
with open(OUTPUT_DIR / "transformations.json", "w", encoding="utf-8") as f:
    json.dump(transformations, f, indent=2, ensure_ascii=False)

# Flat CSV export
with open(OUTPUT_DIR / "transformations.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["doc_title", "template_name", "output", "model", "tokens_used", "duration_s"])
    writer.writeheader()
    writer.writerows(transformations)

# Save analysis
with open(OUTPUT_DIR / "analysis.json", "w", encoding="utf-8") as f:
    json.dump(analysis, f, indent=2, ensure_ascii=False)

# Save manifest
with open(OUTPUT_DIR / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

# Generate markdown report
report_lines = [
    f"# Bulk Research Report",
    f"Generated: {time.strftime('%Y-%m-%d %H:%M')}",
    f"Documents: {len(documents)} | Transformations: {len(transformations)} | Chunks: {len(all_chunks)}",
    "",
    "## Document Manifest",
    "| # | Type | Title |",
    "|---|------|-------|",
]
for i, entry in enumerate(manifest):
    report_lines.append(f"| {i+1} | {entry['type']} | {entry['title'][:60]} |")

report_lines.append("\n## Cross-Document Analysis")
for name, text in analysis.items():
    report_lines.append(f"\n### {name}\n")
    report_lines.append(text)

report_lines.append(f"\n## Search Index\n- Chunks: {len(all_chunks)}\n- Dimensions: {dimension}")

report_path = OUTPUT_DIR / "research_report.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")

saved_files = list(OUTPUT_DIR.glob("*"))
total_size = sum(f.stat().st_size for f in saved_files if f.is_file())
print(f"Saved {len(saved_files)} files to {OUTPUT_DIR} ({total_size:,} bytes):")
for f in sorted(saved_files):
    if f.is_file():
        print(f"  {f.name:30s} {f.stat().st_size:>10,} B")

In [ ]:
# Podcast Script Generation

SPEAKERS = [
    {"name": "Alex", "role": "Host", "style": "Curious, asks clarifying questions, guides the conversation"},
    {"name": "Dr. Chen", "role": "Expert", "style": "Authoritative, cites specific findings, provides depth"},
    {"name": "Sam", "role": "Skeptic", "style": "Challenges assumptions, plays devil's advocate, asks 'so what?'"},
]

# Build context from analysis
findings_context = "\n\n".join(
    f"### {name}\n{text[:3000]}" for name, text in analysis.items()
)

script_prompt = (
    f"Generate a podcast discussion script about these research findings.\n"
    f"Speakers:\n"
    + "\n".join(f"- {s['name']} ({s['role']}): {s['style']}" for s in SPEAKERS)
    + f"\n\nResearch findings:\n{findings_context[:20000]}\n\n"
    f"Format as a JSON array of objects: [{{\"speaker\": \"Name\", \"text\": \"dialogue\"}}]\n"
    f"Generate ~30-40 segments for roughly 10 minutes of dialogue. "
    f"Make it conversational, insightful, and accessible. "
    f"Start with an introduction and end with key takeaways."
)

script_raw = llm_client.chat([{"role": "user", "content": script_prompt}])

# Parse JSON from response (handle markdown code fences)
script_text = script_raw.strip()
if "```" in script_text:
    script_text = script_text.split("```")[1]
    if script_text.startswith("json"):
        script_text = script_text[4:]
podcast_script = json.loads(script_text.strip())

print(f"Generated {len(podcast_script)} dialogue segments\n")
print("Preview:")
for seg in podcast_script[:5]:
    print(f"  {seg['speaker']}: {seg['text'][:100]}...")
if len(podcast_script) > 5:
    print(f"  ... and {len(podcast_script) - 5} more segments")

# Save script
with open(OUTPUT_DIR / "podcast_script.json", "w", encoding="utf-8") as f:
    json.dump(podcast_script, f, indent=2, ensure_ascii=False)

In [ ]:
# Podcast Audio Generation
from pydub import AudioSegment
import io

# Voice mapping per speaker
VOICE_MAP = {
    "Alex": "alloy",
    "Dr. Chen": "onyx",
    "Sam": "nova",
}

tts_client = OpenAI(
    api_key=llm_config.api_key,
    base_url=llm_config.base_url,
)


def generate_tts_segment(text: str, voice: str) -> AudioSegment:
    """Generate audio for a single text segment."""
    if TTS_PROVIDER == "openai":
        response = tts_client.audio.speech.create(
            model="tts-1",
            voice=voice,
            input=text,
        )
        audio_bytes = io.BytesIO(response.content)
        return AudioSegment.from_mp3(audio_bytes)
    else:
        raise ValueError(f"Unsupported TTS provider: {TTS_PROVIDER}. Use 'openai'.")


silence = AudioSegment.silent(duration=500)
podcast_audio = AudioSegment.empty()

for seg in tqdm(podcast_script, desc="Generating audio"):
    voice = VOICE_MAP.get(seg["speaker"], TTS_VOICE)
    try:
        audio_seg = generate_tts_segment(seg["text"], voice)
        podcast_audio += audio_seg + silence
    except Exception as e:
        print(f"  Warning: TTS failed for segment ({seg['speaker']}): {e}")

# Export
podcast_path = OUTPUT_DIR / "research_podcast.mp3"
podcast_audio.export(str(podcast_path), format="mp3")

duration_min = len(podcast_audio) / 1000 / 60
file_size = podcast_path.stat().st_size
print(f"\nPodcast saved: {podcast_path}")
print(f"Duration: {duration_min:.1f} minutes")
print(f"Size: {file_size:,} bytes ({file_size / 1024 / 1024:.1f} MB)")
print(f"Segments: {len(podcast_script)}")

## Usage Guide

### Provider Configuration (`.env`)
```bash
# Pick one LLM provider:
ANTHROPIC_API_KEY=sk-ant-...       # Anthropic (Claude)
OPENAI_API_KEY=sk-...              # OpenAI
GEMINI_API_KEY=...                 # Google Gemini
# Or use local Ollama (no key needed, just set CHAT_MODEL=llama3)

CHAT_MODEL=gpt-4o-mini             # Override default model
EMBEDDING_MODEL=text-embedding-3-small
TTS_PROVIDER=openai
TTS_VOICE=alloy
```

### Adding Source Files
1. Drop PDFs, `.txt`, `.md`, `.html` files into `research_input/`
2. For URLs, create `research_input/urls.csv` with columns: `url,title`
3. Re-run cells 3-4 to discover and extract

### Custom Transformation Templates
Add to the `templates` list in Cell 5:
```python
templates.append(TransformationTemplate(
    name="comparison",
    title="Method Comparison",
    description="Compare methods across related work",
    prompt="Compare the methods in this paper to standard approaches...\n\n{text}"
))
```

### Interactive Search
After running cells 7-8, use interactively in new cells:
```python
results = search("transformer attention mechanism", top_k=10)
answer = ask("What are the main approaches to reducing transformer compute cost?")
print(answer["answer"])
```

### Podcast Customization
Edit `SPEAKERS` in Cell 11 to change personas, and `VOICE_MAP` in Cell 12 for voice assignments.
Available OpenAI voices: `alloy`, `echo`, `fable`, `onyx`, `nova`, `shimmer`.

### Output Directory Structure
```
research_output/
├── manifest.json           # Source file inventory
├── transformations.json    # All AI extractions
├── transformations.csv     # Flat export
├── analysis.json           # Cross-document synthesis
├── research_report.md      # Markdown summary
├── search_index.faiss      # Vector search index
├── podcast_script.json     # Dialogue script
└── research_podcast.mp3    # Audio podcast
```

In [ ]:
# Cleanup (optional — uncomment what you need)
import shutil

# Free FAISS index from memory
# del index, emb_matrix, all_embeddings

# Delete temp files
# shutil.rmtree(OUTPUT_DIR / "temp", ignore_errors=True)

# Disk usage summary
total = sum(f.stat().st_size for f in OUTPUT_DIR.rglob("*") if f.is_file())
print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Total disk usage: {total:,} bytes ({total / 1024 / 1024:.1f} MB)")
for f in sorted(OUTPUT_DIR.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(OUTPUT_DIR)!s:35s} {f.stat().st_size:>10,} B")